In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q diffusers transformers accelerate datasets peft bitsandbytes
!pip install -q git+https://github.com/huggingface/diffusers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.1/516.1 kB 14.4 MB/s eta 0:00:00


In [3]:
!git clone https://github.com/huggingface/diffusers
%cd diffusers/examples/text_to_image
!pip install -r requirements.txt

Cloning into 'diffusers'...
remote: Enumerating objects: 123590, done.
remote: Counting objects: 100% (601/601), done.
remote: Compressing objects: 100% (378/378), done.
remote: Total 123590 (delta 445), reused 227 (delta 222), pack-reused 122989 (from 4)
Receiving objects: 100% (123590/123590), 97.50 MiB | 13.85 MiB/s, done.
Resolving deltas: 100% (92021/92021), done.
/content/diffusers/examples/text_to_image
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00


In [ ]:
import os

sample = []

for root, _, files in os.walk("/content/drive/MyDrive/p2_512"):
    for f in files:
        if f.endswith(".txt"):
            sample.append(os.path.join(root, f))
            break

print("Sample caption file:")
print(sample[0])

with open(sample[0]) as f:
    print(f.read())

Sample caption file:
/content/drive/MyDrive/p2_512/d30/L2/radish_p2_d30_L2_0.txt
radish plant growing in a pot at day rive, top view, realistic


In [ ]:
import os

root = "/content/drive/MyDrive/p2_512"

def extract_day(path):
    for part in path.split("/"):
        if part.startswith("d"):
            return part.replace("d", "")
    return "unknown"

count = 0

for root_dir, _, files in os.walk(root):
    for file in files:
        if file.endswith((".png", ".jpg", ".jpeg")):
            img_path = os.path.join(root_dir, file)
            txt_path = img_path.replace(".png", ".txt").replace(".jpg", ".txt")

            day = extract_day(root_dir)

            caption = f"radish plant growing in a pot at day {day}, top view, realistic"

            with open(txt_path, "w") as f:
                f.write(caption)

            count += 1

print("Created captions:", count)

In [ ]:
import os
import pandas as pd

root = "/content/drive/MyDrive/p2_512"

data = []

def extract_day(path):
    for part in path.split("/"):
        if part.startswith("d"):
            return part.replace("d", "")
    return "unknown"

for root_dir, _, files in os.walk(root):
    for file in files:
        if file.endswith((".png", ".jpg", ".jpeg")):

            full_path = os.path.join(root_dir, file)
            rel_path = os.path.relpath(full_path, root)  # ✅ IMPORTANT

            day = extract_day(root_dir)

            caption = f"radish plant growing in a pot at day {day}, top view, realistic, soil visible"

            data.append({
                "file_name": rel_path,   # ✅ MUST BE THIS
                "text": caption
            })

df = pd.DataFrame(data)

csv_path = os.path.join(root, "metadata.csv")
df.to_csv(csv_path, index=False)

print("✅ FINAL metadata created correctly")

In [4]:
!pip install torchao --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="/content/drive/MyDrive/p2_512" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=2 \
  --gradient_accumulation_steps=4 \
  --learning_rate=1e-4 \
  --max_train_steps=6000 \
  --checkpointing_steps=200 \
  --resume_from_checkpoint="checkpoint-3400" \
  --output_dir="/content/drive/MyDrive/radish_lora_model" \
  --mixed_precision="fp16"

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorc

In [ ]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
import torch

device = "cuda"

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-seg",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to(device)

pipe.enable_model_cpu_offload()

# load your trained LoRA
pipe.load_lora_weights("/content/drive/MyDrive/radish_lora_model/checkpoint-3400")

print("✅ Ready for mask-based generation")

In [ ]:
import re
import random
import glob

# -----------------------------
# EXTRACT DAY FROM PROMPT
# -----------------------------
def get_day_from_prompt(prompt):

    match = re.search(r"day\s*(\d+)", prompt.lower())

    if match:
        return int(match.group(1))

    return None


# -----------------------------
# GET RANDOM MASK FOR THAT DAY
# -----------------------------
def get_mask_for_day(mask_list, day):

    filtered = [m for m in mask_list if f"/d{day}/" in m]

    if len(filtered) == 0:
        return None

    return random.choice(filtered)


# -----------------------------
# LOAD ALL MASKS
# -----------------------------
mask_list = glob.glob(
    "/content/drive/MyDrive/p2_512_masks/**/*.png",
    recursive=True
)

print("✅ Total masks:", len(mask_list))

In [ ]:
import cv2
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
import glob

p2_masks = glob.glob("/content/drive/MyDrive/p2_512_masks/**/*.*", recursive=True)
p3_masks = glob.glob("/content/drive/MyDrive/p3_maskssss/**/*.*", recursive=True)

all_masks = p2_masks + p3_masks

print("Total masks:", len(all_masks))

In [ ]:
import random

def get_masks_for_day(mask_list, day):
    day_str1 = f"d{day}"
    day_str2 = f"day_{day}"

    return [p for p in mask_list if (day_str1 in p or day_str2 in p)]


def get_random_mask(mask_list, day):
    candidates = get_masks_for_day(mask_list, day)

    if len(candidates) == 0:
        return None

    return random.choice(candidates)

In [ ]:
import re

def get_day_from_prompt(prompt):
    match = re.search(r"day\s*(\d+)", prompt.lower())

    if match:
        return int(match.group(1))

    return None

In [ ]:
from PIL import Image
import cv2
import matplotlib.pyplot as plt


prompt="radish plant in white pot with soil at day 26"

day = get_day_from_prompt(prompt)
print("Detected day:", day)

mask_path = get_random_mask(all_masks, day)

if mask_path is None:
    raise ValueError("No mask found — check dataset naming")

print("Using mask:", mask_path)

mask = cv2.imread(mask_path, 0)
mask = cv2.resize(mask, (512, 512))
mask = Image.fromarray(mask).convert("RGB")

import matplotlib.pyplot as plt
plt.imshow(mask)
plt.title("Random Mask")
plt.axis("off")

In [ ]:
def get_day_from_prompt(prompt):
    match = re.search(r"day\s*(\d+)", prompt.lower())
    if match:
        return int(match.group(1))
    return None

In [ ]:
import re

def extract_day_from_path(path):
    # find ALL d<number> patterns
    matches = re.findall(r"d(\d+)", path)

    if len(matches) == 0:
        return None

    # take first valid occurrence
    return int(matches[0])


def get_mask_for_day(mask_list, day):

    filtered = []

    for m in mask_list:
        file_day = extract_day_from_path(m)

        if file_day is not None and file_day == day:
            filtered.append(m)

    if len(filtered) == 0:
        print("❌ No mask found for day:", day)
        print("👉 Available sample paths:")
        print(mask_list[:5])
        return None

    print(f"✅ Found {len(filtered)} masks for day {day}")
    return sorted(filtered)[0]

In [ ]:
image = pipe(
    prompt= prompt,
    negative_prompt="distorted, deformed, extra leaves, roots visible,non green leaves",
    image=mask,   # 🔥 THIS IS THE KEY
    guidance_scale=10,
    num_inference_steps=50,
    controlnet_conditioning_scale=1.8
).images[0]
plt.imshow(image)
plt.axis("off")

In [ ]:
import os
from PIL import Image

# ===== OUTPUT FOLDER =====
save_dir = "/content/drive/MyDrive/generated_radish26"

os.makedirs(save_dir, exist_ok=True)

# ===== GENERATE 20 IMAGES =====
for i in range(20):

    image = pipe(
        prompt=prompt,
        negative_prompt="distorted, deformed, extra leaves, roots visible",
        image=mask,
        guidance_scale=10,
        num_inference_steps=50,
        controlnet_conditioning_scale=1.8
    ).images[0]

    # filename using detected day
    save_path = os.path.join(save_dir, f"generated_day_{day}_{i+1}.png")

    image.save(save_path)

    print(f"Saved: {save_path}")

print("All images generated and saved successfully!")

In [ ]:
!pip install clean-fid

In [ ]:
# ======================================================
# INSTALL (run once)
# ======================================================
# !pip install clean-fid

# ======================================================
# IMPORTS
# ======================================================
import os
import shutil
from cleanfid import fid

# ======================================================
# PATHS
# ======================================================
real_root = "/content/drive/MyDrive/p2_512"
gen_root = "/content/drive/MyDrive/generated_radish26"

# ======================================================
# DETECT GENERATED DAYS
# Example:
# generated_day_18_1.png -> 18
# ======================================================
days = sorted(list(set([
    f.split("_")[2]
    for f in os.listdir(gen_root)
    if f.endswith(".png")
])))

print("Detected days:", days)

# ======================================================
# STORE RESULTS
# ======================================================
results = {}

# ======================================================
# LOOP THROUGH DAYS
# ======================================================
for day in days:

    print(f"\nProcessing Day {day}")

    # --------------------------------------------------
    # REAL DATASET DAY PATH
    # Example:
    # p2_512/d18
    # --------------------------------------------------
    day_path = os.path.join(real_root, f"d{day}")

    if not os.path.exists(day_path):

        print(f"Day folder missing: {day_path}")
        continue

    # --------------------------------------------------
    # TEMP FOLDERS
    # --------------------------------------------------
    real_day_folder = f"/content/temp_real_day_{day}"
    gen_day_folder  = f"/content/temp_gen_day_{day}"

    # remove old temp folders if exist
    if os.path.exists(real_day_folder):
        shutil.rmtree(real_day_folder)

    if os.path.exists(gen_day_folder):
        shutil.rmtree(gen_day_folder)

    os.makedirs(real_day_folder, exist_ok=True)
    os.makedirs(gen_day_folder, exist_ok=True)

    # ==================================================
    # COPY REAL IMAGES
    # Structure:
    # d18/level_1/*.png
    # d18/level_2/*.png
    # ==================================================
    for level in os.listdir(day_path):

        level_path = os.path.join(day_path, level)

        if os.path.isdir(level_path):

            for file in os.listdir(level_path):

                if file.endswith((".png", ".jpg", ".jpeg")):

                    src = os.path.join(level_path, file)
                    dst = os.path.join(real_day_folder, file)

                    shutil.copy(src, dst)

    # ==================================================
    # COPY GENERATED IMAGES
    # ==================================================
    for file in os.listdir(gen_root):

        if f"day_{day}" in file:

            src = os.path.join(gen_root, file)
            dst = os.path.join(gen_day_folder, file)

            shutil.copy(src, dst)

    # ==================================================
    # COUNT IMAGES
    # ==================================================
    real_count = len(os.listdir(real_day_folder))
    gen_count  = len(os.listdir(gen_day_folder))

    print(f"Real images: {real_count}")
    print(f"Generated images: {gen_count}")

    # ==================================================
    # COMPUTE KID
    # ==================================================
    try:

        kid_score = fid.compute_kid(
            real_day_folder,
            gen_day_folder
        )

        results[day] = kid_score

        print(f"Day {day} → KID: {kid_score}")

    except Exception as e:

        print(f"Error computing KID for day {day}")
        print(e)

# ======================================================
# FINAL RESULTS
# ======================================================
print("\n========== FINAL KID SCORES ==========")

for day, score in results.items():

    print(f"Day {day}: {score}")